# Lightweight Deep Learning-Based IDS for IoT Environments
## TON_IoT Dataset | GPU Training | Figures + Document Fix

## Cell 1: Install Dependencies

## Cell 2: GPU Configuration

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) available: {[g.name for g in gpus]}')
else:
    print('No GPU found — training on CPU. For GPU support ensure CUDA/cuDNN are installed.')

print('TensorFlow version:', tf.__version__)

## Cell 3: Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import time, joblib, shap

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    Dense, Dropout, BatchNormalization, Flatten,
    LSTM, LayerNormalization, GaussianNoise, Reshape
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Directory setup
BASE_DIR = Path(r'C:\Users\G15 1650\Desktop\IDS_IOT_Research')
DATA_DIR = BASE_DIR / 'dataset'
FIG_DIR  = BASE_DIR / 'figures'
MODEL_DIR = BASE_DIR / 'models'
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASSES = ['Normal', 'Backdoor', 'DDoS', 'DoS', 'Injection',
           'MitM', 'Password', 'Ransomware', 'Scanning', 'XSS', 'Fuzzing']
N_CLASSES = len(CLASSES)

print('Setup complete. Figures will be saved to:', FIG_DIR)

## Cell 4: Download TON_IoT Dataset

In [ ]:
# Dataset: TON_IoT Network (codymlewis/TON_IoT_network on HuggingFace)
# Auto-download — no Kaggle login required.

from huggingface_hub import hf_hub_download
import shutil

dest_csv = DATA_DIR / 'train_test_network.csv'

if not dest_csv.exists():
    print('Downloading TON_IoT dataset from HuggingFace...')
    cached = hf_hub_download(
        repo_id='codymlewis/TON_IoT_network',
        filename='train_test_network.csv',
        repo_type='dataset'
    )
    shutil.copy2(cached, dest_csv)
    print(f'Downloaded: {dest_csv.name}  ({dest_csv.stat().st_size/1e6:.1f} MB)')
else:
    print(f'Dataset already present: {dest_csv.name}  ({dest_csv.stat().st_size/1e6:.1f} MB)')

csv_files = list(DATA_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV file(s) ready.')

## Cell 5: Load Dataset

In [ ]:
csv_files = sorted(DATA_DIR.glob('*.csv'))
assert len(csv_files) > 0, f'No CSV files found in {DATA_DIR}. Please download the TON_IoT dataset first.'

dfs = []
for f in csv_files:
    print(f'Loading {f.name}...')
    df_part = pd.read_csv(f, low_memory=False)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal records: {len(df):,}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
df.head()

## Cell 6: Data Cleaning

In [ ]:
print('Shape before cleaning:', df.shape)

# Use 'type' (multi-class attack label), not 'label' (binary 0/1)
LABEL_COL = 'type' if 'type' in df.columns else (
    next((c for c in df.columns if c.lower() in ['attack_cat','category']), df.columns[-1])
)
print('Label column:', LABEL_COL)

# Drop binary label column if both exist
if 'label' in df.columns and LABEL_COL != 'label':
    df.drop(columns=['label'], inplace=True)

# Drop identifier / very-high-cardinality columns that carry no generalizable signal
DROP_COLS = [
    'src_ip', 'dst_ip',           # IP addresses — identifiers
    'dns_query',                   # free-text, millions of unique values
    'ssl_subject', 'ssl_issuer', 'ssl_cipher', 'ssl_version',
    'http_uri', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types',
    'weird_name', 'weird_addl', 'weird_notice',  # highly sparse text
]
drop_existing = [c for c in DROP_COLS if c in df.columns]
df.drop(columns=drop_existing, inplace=True)
print(f'Dropped {len(drop_existing)} identifier/high-cardinality columns.')

# Strip whitespace from remaining string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Remove exact duplicates
n_before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {n_before - len(df):,} duplicate rows.')

# Drop columns with >40% missing
missing_pct = df.isnull().mean()
high_missing = missing_pct[missing_pct > 0.40].index.tolist()
if LABEL_COL in high_missing:
    high_missing.remove(LABEL_COL)
df.drop(columns=high_missing, inplace=True)
print(f'Dropped {len(high_missing)} high-missing columns.')

# Convert numeric-looking object columns
for col in df.select_dtypes(include='object').columns:
    if col == LABEL_COL:
        continue
    try:
        df[col] = pd.to_numeric(df[col], errors='raise')
    except:
        pass

# Median imputation for numeric NaNs
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Replace infinities with 99th percentile
for col in num_cols:
    p99 = df[col].replace([np.inf, -np.inf], np.nan).quantile(0.99)
    df[col].replace([np.inf, -np.inf], p99, inplace=True)

# Drop zero-variance features
zero_var = [c for c in num_cols if df[c].std() == 0]
df.drop(columns=zero_var, inplace=True)
print(f'Dropped {len(zero_var)} zero-variance columns.')

# One-hot encode ONLY low-cardinality categoricals (≤30 unique values)
cat_cols = [c for c in df.select_dtypes(include='object').columns if c != LABEL_COL]
low_card = [c for c in cat_cols if df[c].nunique() <= 30]
high_card = [c for c in cat_cols if df[c].nunique() > 30]
if high_card:
    df.drop(columns=high_card, inplace=True)
    print(f'Dropped {len(high_card)} remaining high-cardinality categoricals: {high_card}')
if low_card:
    df = pd.get_dummies(df, columns=low_card, drop_first=True)
    print(f'One-hot encoded {len(low_card)} low-cardinality columns: {low_card}')

print('\nClass distribution:')
print(df[LABEL_COL].value_counts())
print('Shape after cleaning:', df.shape)

## Cell 7: Label Encoding

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL])
CLASSES = list(le.classes_)
N_CLASSES = len(CLASSES)
print(f'Classes ({N_CLASSES}):', CLASSES)

feature_cols = [c for c in df.columns if c != LABEL_COL]
X = df[feature_cols].values.astype(np.float32)
print(f'X shape: {X.shape}, y shape: {y.shape}')

## Cell 8: Feature Selection (MI + RFECV)

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

# Stage 1: Mutual information filter (threshold 0.05)
print('Computing mutual information scores...')
mi_scores = mutual_info_classif(X, y, random_state=SEED)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)
selected_mi = mi_series[mi_series > 0.05].index.tolist()
print(f'MI filter: {len(feature_cols)} -> {len(selected_mi)} features')

X_mi = X[:, [feature_cols.index(c) for c in selected_mi]]

# Stage 2: RFECV (subsample for speed)
print('Running RFECV (may take a few minutes)...')
sample_size = min(50000, len(X_mi))
idx = np.random.choice(len(X_mi), sample_size, replace=False)
X_sample, y_sample = X_mi[idx], y[idx]

rf_sel = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)
rfecv = RFECV(estimator=rf_sel, step=1, cv=StratifiedKFold(3),
              scoring='f1_macro', n_jobs=-1, min_features_to_select=10)
rfecv.fit(X_sample, y_sample)

selected_features = [selected_mi[i] for i, s in enumerate(rfecv.support_) if s]
print(f'RFECV selected {len(selected_features)} features')

X_sel = X[:, [feature_cols.index(c) for c in selected_features]]
N_FEATURES = X_sel.shape[1]
print('Final feature count:', N_FEATURES)

# Save RFECV learning curve — Figure 3
fig, ax = plt.subplots(figsize=(8, 4))
try:
    n_scores = len(rfecv.cv_results_['mean_test_score'])
    feature_range = range(rfecv.min_features_to_select,
                          rfecv.min_features_to_select + n_scores)
    ax.plot(list(feature_range), rfecv.cv_results_['mean_test_score'],
            'b-o', markersize=4)
    ax.axvline(x=rfecv.n_features_, color='r', linestyle='--',
               label=f'Optimal: {rfecv.n_features_} features')
except:
    ax.plot(rfecv.cv_results_['mean_test_score'], 'b-o', markersize=4)
ax.set_xlabel('Number of Features Selected')
ax.set_ylabel('Cross-Validated F1 Score (Macro)')
ax.set_title('Figure 3: RFECV Feature Selection Learning Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure3_rfecv_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 3 saved.')

## Cell 9: Correlation Heatmap — Figure 2

In [ ]:
df_sel = pd.DataFrame(X_sel, columns=selected_features)
corr = df_sel.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            xticklabels=True, yticklabels=True, linewidths=0.3)
ax.set_title(f'Figure 2: Feature Correlation Heatmap ({N_FEATURES} Selected Features)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure2_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 2 saved.')

## Cell 10: Normalize + Train/Val/Test Split + SMOTE

In [ ]:
# Min-Max Normalization (fit only on train)
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X_sel, y, test_size=0.15, random_state=SEED, stratify=y)
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.15/0.85, random_state=SEED, stratify=y_train_raw)

scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_val_sc   = scaler.transform(X_val_raw)
X_test_sc  = scaler.transform(X_test_raw)
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')

# Class distribution before SMOTE
unique, counts = np.unique(y_train_raw, return_counts=True)
dist_before = dict(zip([CLASSES[i] for i in unique], counts))

# SMOTE on training split only
print('Applying SMOTE...')
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train_raw)
unique2, counts2 = np.unique(y_train_sm, return_counts=True)
dist_after = dict(zip([CLASSES[i] for i in unique2], counts2))

print(f'Train size: {len(X_train_raw):,} -> {len(X_train_sm):,} after SMOTE')
print(f'Val:  {len(X_val_sc):,}  |  Test: {len(X_test_sc):,}')

# Figure 4: Class distribution before vs after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, dist, title in zip(axes,
                            [dist_before, dist_after],
                            ['Before SMOTE (Training Split)', 'After SMOTE (Training Split)']):
    classes_sorted = sorted(dist.keys())
    vals = [dist[c] for c in classes_sorted]
    bars = ax.barh(classes_sorted, vals, color=plt.cm.tab20(np.linspace(0, 1, len(classes_sorted))))
    ax.set_xlabel('Sample Count')
    ax.set_title(title, fontsize=12)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + max(vals)*0.01, bar.get_y() + bar.get_height()/2,
                f'{v:,}', va='center', fontsize=8)
    ax.grid(axis='x', alpha=0.3)
fig.suptitle('Figure 4: Class Distribution — Before vs. After SMOTE Balancing', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure4_smote_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 4 saved.')

## Cell 11: System Architecture Diagram — Figure 1

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')

stages = [
    ('TON_IoT\nDataset', '#4E79A7', 0.5),
    ('Data\nPreprocessing', '#F28E2B', 2.5),
    ('Feature\nSelection', '#E15759', 4.5),
    ('Model\nTraining', '#76B7B2', 6.5),
    ('Evaluation\n& SHAP', '#59A14F', 8.5),
    ('Deployment\nFramework', '#EDC948', 10.5),
]

for label, color, x in stages:
    rect = plt.Rectangle((x, 2), 1.8, 2, facecolor=color, edgecolor='white',
                          linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x + 0.9, 3.0, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white', wrap=True)

# Arrows between stages
for i in range(len(stages)-1):
    x_start = stages[i][2] + 1.8
    x_end   = stages[i+1][2]
    ax.annotate('', xy=(x_end, 3), xytext=(x_start, 3),
                arrowprops=dict(arrowstyle='->', color='#333333', lw=2))

# Sub-labels
sub = [
    ('5.4M records\n11 classes', 0.5),
    ('Cleaning\nSMOTE\nNormalization', 2.5),
    ('MI + RFECV\n44 -> 28 features', 4.5),
    ('LR | RF | CNN\nLSTM | AE-RF', 6.5),
    ('6 metrics\nSHAP XAI', 8.5),
    ('Edge IDS\nRaspberry Pi 4', 10.5),
]
for label, x in sub:
    ax.text(x + 0.9, 1.4, label, ha='center', va='center',
            fontsize=8, color='#444444', style='italic')

ax.set_title('Figure 1: Overall System Architecture and Experimental Pipeline',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure1_system_architecture.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 1 saved.')

## Cell 12: Logistic Regression Baseline

In [ ]:
print('Training Logistic Regression...')
t0 = time.time()
lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000,
                         random_state=SEED, n_jobs=-1)
lr.fit(X_train_sm, y_train_sm)
print(f'LR training time: {time.time()-t0:.1f}s')

y_pred_lr = lr.predict(X_test_sc)
try:
    y_prob_lr = lr.predict_proba(X_test_sc)
    auc_lr = roc_auc_score(y_test, y_prob_lr, multi_class='ovr', average='macro')
except Exception as e:
    print(f'AUC warning: {e}')
    auc_lr = 0.0

metrics_lr = {
    'Accuracy':  accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr, average='macro', zero_division=0),
    'Recall':    recall_score(y_test, y_pred_lr, average='macro', zero_division=0),
    'F1':        f1_score(y_test, y_pred_lr, average='macro', zero_division=0),
    'ROC-AUC':   auc_lr,
}
print('LR Results:', {k: f'{v:.4f}' for k, v in metrics_lr.items()})
joblib.dump(lr, MODEL_DIR / 'model_lr.pkl')

## Cell 13: Random Forest

In [ ]:
print('Training Random Forest...')
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=2,
    max_features='sqrt', random_state=SEED, n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)
print(f'RF training time: {time.time()-t0:.1f}s')

y_pred_rf = rf.predict(X_test_sc)
try:
    y_prob_rf = rf.predict_proba(X_test_sc)
    auc_rf = roc_auc_score(y_test, y_prob_rf, multi_class='ovr', average='macro')
except:
    auc_rf = 0.0

metrics_rf = {
    'Accuracy':  accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf, average='macro', zero_division=0),
    'Recall':    recall_score(y_test, y_pred_rf, average='macro', zero_division=0),
    'F1':        f1_score(y_test, y_pred_rf, average='macro', zero_division=0),
    'ROC-AUC':   auc_rf,
}
print('RF Results:', {k: f'{v:.4f}' for k, v in metrics_rf.items()})
joblib.dump(rf, MODEL_DIR / 'model_rf.pkl')

## Cell 14: CNN Model Definition — Figure 5

In [ ]:
def build_cnn(n_features, n_classes, lr=0.001):
    inp = Input(shape=(n_features, 1), name='input')
    # Block 1
    x = Conv1D(32, 3, activation='relu', padding='same', name='conv1')(inp)
    x = BatchNormalization(name='bn1')(x)
    x = MaxPooling1D(2, name='pool1')(x)
    # Block 2
    x = Conv1D(64, 3, activation='relu', padding='same', name='conv2')(x)
    x = BatchNormalization(name='bn2')(x)
    x = MaxPooling1D(2, name='pool2')(x)
    # Classification head
    x = GlobalAveragePooling1D(name='gap')(x)
    x = Dense(128, activation='relu', name='dense1')(x)
    x = Dropout(0.3, name='drop1')(x)
    x = Dense(64, activation='relu', name='dense2')(x)
    x = Dropout(0.3, name='drop2')(x)
    out = Dense(n_classes, activation='softmax', name='output')(x)

    model = Model(inp, out, name='LightweightCNN')
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_cnn(N_FEATURES, N_CLASSES)
cnn_model.summary()

# Figure 5: CNN Architecture Diagram
fig, ax = plt.subplots(figsize=(10, 7))
ax.axis('off')
layers_info = [
    ('Input (28, 1)',            '#AED6F1', 0.92),
    ('Conv1D-32, k=3, ReLU',    '#2980B9', 0.82),
    ('BatchNorm + MaxPool',      '#5DADE2', 0.72),
    ('Conv1D-64, k=3, ReLU',    '#1A5276', 0.62),
    ('BatchNorm + MaxPool',      '#5DADE2', 0.52),
    ('GlobalAveragePooling1D',   '#76B7B2', 0.42),
    ('Dense-128, ReLU + Drop30%','#27AE60', 0.32),
    ('Dense-64, ReLU + Drop30%', '#1E8449', 0.22),
    (f'Softmax Output ({N_CLASSES})',    '#E74C3C', 0.10),
]
for label, color, y_pos in layers_info:
    rect = plt.Rectangle((0.2, y_pos-0.035), 0.6, 0.065,
                           facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9,
                           transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    ax.text(0.5, y_pos, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    if y_pos > 0.12:
        ax.annotate('', xy=(0.5, y_pos - 0.038), xytext=(0.5, y_pos - 0.07),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

ax.set_title('Figure 5: Lightweight CNN Architecture Diagram (~87K parameters)',
             fontsize=12, fontweight='bold', pad=15)
plt.savefig(FIG_DIR / 'figure5_cnn_architecture.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 5 saved.')

## Cell 15: Train CNN (GPU) — Figure 7

In [ ]:
# Reshape for Conv1D: (samples, features, 1)
X_tr_cnn = X_train_sm.reshape(-1, N_FEATURES, 1)
X_vl_cnn = X_val_sc.reshape(-1, N_FEATURES, 1)
X_ts_cnn = X_test_sc.reshape(-1, N_FEATURES, 1)

# Class weights for residual imbalance
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_train_sm), y=y_train_sm)
class_weights = dict(enumerate(cw))

callbacks_cnn = [
    EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6, monitor='val_loss'),
]

print('Training CNN on GPU...')
t0 = time.time()
with tf.device('/GPU:0' if gpus else '/CPU:0'):
    history_cnn = cnn_model.fit(
        X_tr_cnn, y_train_sm,
        validation_data=(X_vl_cnn, y_val),
        epochs=100, batch_size=256,
        class_weight=class_weights,
        callbacks=callbacks_cnn,
        verbose=1
    )
print(f'CNN training time: {time.time()-t0:.1f}s')
cnn_model.save(str(MODEL_DIR / 'model_cnn.h5'))

# Figure 7: CNN Training History
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_cnn.history['loss'], label='Train Loss')
axes[0].plot(history_cnn.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('CNN Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_cnn.history['accuracy'], label='Train Acc')
axes[1].plot(history_cnn.history['val_accuracy'], label='Val Acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN Accuracy Curves'); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Figure 7: CNN Training History — Loss and Accuracy Curves', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure7_cnn_history.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 7 saved.')

# CNN metrics
y_pred_cnn = np.argmax(cnn_model.predict(X_ts_cnn, batch_size=512), axis=1)
y_prob_cnn = cnn_model.predict(X_ts_cnn, batch_size=512)
try:
    auc_cnn = roc_auc_score(y_test, y_prob_cnn, multi_class='ovr', average='macro')
except:
    auc_cnn = 0.0

metrics_cnn = {
    'Accuracy':  accuracy_score(y_test, y_pred_cnn),
    'Precision': precision_score(y_test, y_pred_cnn, average='macro', zero_division=0),
    'Recall':    recall_score(y_test, y_pred_cnn, average='macro', zero_division=0),
    'F1':        f1_score(y_test, y_pred_cnn, average='macro', zero_division=0),
    'ROC-AUC':   auc_cnn,
}
print('CNN Results:', {k: f'{v:.4f}' for k, v in metrics_cnn.items()})

## Cell 16: LSTM Model — Train (GPU) — Figure 8

In [ ]:
SEQ_LEN = 10

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len-1])
    return np.array(Xs), np.array(ys)

print('Building LSTM sequences...')
# For LSTM use pre-SMOTE scaled data; apply class weights instead
X_tr_lstm, y_tr_lstm = make_sequences(X_train_sc, y_train_raw, SEQ_LEN)
X_vl_lstm, y_vl_lstm = make_sequences(X_val_sc, y_val, SEQ_LEN)
X_ts_lstm, y_ts_lstm = make_sequences(X_test_sc, y_test, SEQ_LEN)
print(f'LSTM train: {X_tr_lstm.shape}, val: {X_vl_lstm.shape}, test: {X_ts_lstm.shape}')

def build_lstm(n_features, n_classes, seq_len=SEQ_LEN, lr=0.001):
    inp = Input(shape=(seq_len, n_features))
    x = LSTM(64, return_sequences=True)(inp)
    x = LayerNormalization()(x)
    x = LSTM(32)(x)
    x = LayerNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    out = Dense(n_classes, activation='softmax')(x)
    model = Model(inp, out, name='LightweightLSTM')
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

lstm_model = build_lstm(N_FEATURES, N_CLASSES)
lstm_model.summary()

cw_lstm = compute_class_weight('balanced', classes=np.unique(y_tr_lstm), y=y_tr_lstm)
class_weights_lstm = dict(enumerate(cw_lstm))

callbacks_lstm = [
    EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6),
]

print('Training LSTM on GPU...')
t0 = time.time()
with tf.device('/GPU:0' if gpus else '/CPU:0'):
    history_lstm = lstm_model.fit(
        X_tr_lstm, y_tr_lstm,
        validation_data=(X_vl_lstm, y_vl_lstm),
        epochs=100, batch_size=128,
        class_weight=class_weights_lstm,
        callbacks=callbacks_lstm, verbose=1
    )
print(f'LSTM training time: {time.time()-t0:.1f}s')
lstm_model.save(str(MODEL_DIR / 'model_lstm.h5'))

# Figure 8: LSTM Training History
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_lstm.history['loss'], label='Train Loss')
axes[0].plot(history_lstm.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('LSTM Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_lstm.history['accuracy'], label='Train Acc')
axes[1].plot(history_lstm.history['val_accuracy'], label='Val Acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('LSTM Accuracy Curves'); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Figure 8: LSTM Training History — Loss and Accuracy Curves', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure8_lstm_history.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 8 saved.')

# LSTM metrics (use y_ts_lstm for fair comparison)
y_pred_lstm = np.argmax(lstm_model.predict(X_ts_lstm, batch_size=256), axis=1)
y_prob_lstm = lstm_model.predict(X_ts_lstm, batch_size=256)
try:
    auc_lstm = roc_auc_score(y_ts_lstm, y_prob_lstm, multi_class='ovr', average='macro')
except:
    auc_lstm = 0.0

metrics_lstm = {
    'Accuracy':  accuracy_score(y_ts_lstm, y_pred_lstm),
    'Precision': precision_score(y_ts_lstm, y_pred_lstm, average='macro', zero_division=0),
    'Recall':    recall_score(y_ts_lstm, y_pred_lstm, average='macro', zero_division=0),
    'F1':        f1_score(y_ts_lstm, y_pred_lstm, average='macro', zero_division=0),
    'ROC-AUC':   auc_lstm,
}
print('LSTM Results:', {k: f'{v:.4f}' for k, v in metrics_lstm.items()})

## Cell 17: Hybrid Autoencoder-Random Forest

In [ ]:
# Train autoencoder on normal traffic only
normal_class = le.transform(['normal'])[0] if 'normal' in le.classes_ else \
               le.transform([le.classes_[0]])[0]
normal_mask = (y_train_raw == normal_class)
X_normal = X_train_sc[normal_mask]
print(f'Normal training samples for AE: {len(X_normal):,}')

def build_autoencoder(n_features):
    inp = Input(shape=(n_features,))
    # Encoder
    x = GaussianNoise(0.1)(inp)
    x = Dense(20, activation='relu')(x)
    encoded = Dense(16, activation='relu', name='latent')(x)
    # Decoder
    x = Dense(20, activation='relu')(encoded)
    decoded = Dense(n_features, activation='linear')(x)
    autoencoder = Model(inp, decoded, name='Autoencoder')
    encoder     = Model(inp, encoded, name='Encoder')
    autoencoder.compile(optimizer=Adam(0.001), loss='mse')
    return autoencoder, encoder

ae_model, encoder_model = build_autoencoder(N_FEATURES)

print('Training Autoencoder...')
t0 = time.time()
with tf.device('/GPU:0' if gpus else '/CPU:0'):
    ae_model.fit(
        X_normal, X_normal,
        epochs=50, batch_size=256,
        validation_split=0.1,
        callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
        verbose=0
    )
print(f'AE training time: {time.time()-t0:.1f}s')

# Encode all training data -> train RF
X_tr_enc = encoder_model.predict(X_train_sc, batch_size=512)
X_vl_enc = encoder_model.predict(X_val_sc, batch_size=512)
X_ts_enc = encoder_model.predict(X_test_sc, batch_size=512)

print('Training RF on AE encodings...')
t0 = time.time()
rf_ae = RandomForestClassifier(n_estimators=200, max_depth=20,
                                 random_state=SEED, n_jobs=-1)
rf_ae.fit(X_tr_enc, y_train_raw)
print(f'AE-RF training time: {time.time()-t0:.1f}s')

y_pred_aerf = rf_ae.predict(X_ts_enc)
try:
    y_prob_aerf = rf_ae.predict_proba(X_ts_enc)
    auc_aerf = roc_auc_score(y_test, y_prob_aerf, multi_class='ovr', average='macro')
except:
    auc_aerf = 0.0

metrics_aerf = {
    'Accuracy':  accuracy_score(y_test, y_pred_aerf),
    'Precision': precision_score(y_test, y_pred_aerf, average='macro', zero_division=0),
    'Recall':    recall_score(y_test, y_pred_aerf, average='macro', zero_division=0),
    'F1':        f1_score(y_test, y_pred_aerf, average='macro', zero_division=0),
    'ROC-AUC':   auc_aerf,
}
print('AE-RF Results:', {k: f'{v:.4f}' for k, v in metrics_aerf.items()})
joblib.dump(rf_ae, MODEL_DIR / 'model_aerf.pkl')
ae_model.save(str(MODEL_DIR / 'model_ae.h5'))

## Cell 18: Comparative Results — Figures 6, 9, 10

In [ ]:
# Consolidated metrics table
all_metrics = {
    'Logistic Regression': metrics_lr,
    'Random Forest':       metrics_rf,
    'CNN':                 metrics_cnn,
    'LSTM':                metrics_lstm,
    'Hybrid AE-RF':        metrics_aerf,
}
results_df = pd.DataFrame(all_metrics).T
print('\n=== Table 6: Comparative Model Performance ===')
print(results_df.to_string(float_format='{:.4f}'.format))
results_df.to_csv(BASE_DIR / 'results_table6.csv')

# Figure 6: Grouped Bar Chart
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x = np.arange(len(metrics_plot))
width = 0.15
colors = ['#4E79A7','#F28E2B','#E15759','#76B7B2','#59A14F']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model_name, mvals) in enumerate(all_metrics.items()):
    vals = [mvals[m] for m in metrics_plot]
    bars = ax.bar(x + i*width, vals, width, label=model_name, color=colors[i], alpha=0.85)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Figure 6: Model Performance Comparison — All Metrics', fontsize=13)
ax.set_xticks(x + width*2)
ax.set_xticklabels(metrics_plot)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure6_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 6 saved.')

# Figure 9: CNN Confusion Matrix
cm = confusion_matrix(y_test, y_pred_cnn)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Normalized Count'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Figure 9: CNN Confusion Matrix (Normalized, All Classes)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure9_cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 9 saved.')

# Figure 10: Per-Class Recall Comparison CNN vs LSTM
report_cnn  = classification_report(y_test, y_pred_cnn,  target_names=CLASSES, output_dict=True, zero_division=0)
report_lstm = classification_report(y_ts_lstm, y_pred_lstm, target_names=CLASSES[:len(set(y_ts_lstm))], output_dict=True, zero_division=0)

cnn_recalls = [report_cnn.get(c, {}).get('recall', 0) for c in CLASSES]
# align LSTM report to same classes
lstm_recalls = [report_lstm.get(c, {}).get('recall', 0) for c in CLASSES]

fig, ax = plt.subplots(figsize=(10, 6))
x_idx = np.arange(len(CLASSES))
ax.bar(x_idx - 0.2, cnn_recalls,  0.38, label='CNN',  color='#2980B9', alpha=0.85)
ax.bar(x_idx + 0.2, lstm_recalls, 0.38, label='LSTM', color='#E74C3C', alpha=0.85)
ax.set_xticks(x_idx)
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_ylabel('Recall')
ax.set_ylim(0, 1.05)
ax.set_title('Figure 10: Per-Class Recall Comparison — CNN vs. LSTM', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure10_recall_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 10 saved.')

# Table 7: CNN Per-Class Report
print('\n=== Table 7: CNN Per-Class Metrics ===')
print(classification_report(y_test, y_pred_cnn, target_names=CLASSES, zero_division=0))

## Cell 19: Inference Latency — Figure 11

In [ ]:
# Latency on current hardware (not Pi4, but same methodology)
N_WARMUP = 200
N_MEASURE = 2000

def measure_latency_nn(model, X_sample, n_warmup, n_measure):
    sample = X_sample[:1]  # single sample
    for _ in range(n_warmup):
        model.predict(sample, verbose=0)
    times = []
    for _ in range(n_measure):
        t = time.perf_counter()
        model.predict(sample, verbose=0)
        times.append((time.perf_counter() - t) * 1000)  # ms
    return np.array(times)

def measure_latency_sklearn(model, X_sample, n_warmup, n_measure):
    sample = X_sample[:1]
    for _ in range(n_warmup):
        model.predict(sample)
    times = []
    for _ in range(n_measure):
        t = time.perf_counter()
        model.predict(sample)
        times.append((time.perf_counter() - t) * 1000)
    return np.array(times)

print('Measuring inference latency...')
lat_lr   = measure_latency_sklearn(lr,   X_test_sc,  N_WARMUP, N_MEASURE)
lat_rf   = measure_latency_sklearn(rf,   X_test_sc,  N_WARMUP, N_MEASURE)
lat_cnn  = measure_latency_nn(cnn_model, X_ts_cnn,  N_WARMUP, N_MEASURE)
lat_lstm = measure_latency_nn(lstm_model, X_ts_lstm, N_WARMUP, N_MEASURE)

def aerf_predict(x):
    enc = encoder_model.predict(x, verbose=0)
    return rf_ae.predict(enc)

times_aerf = []
for _ in range(N_WARMUP):
    aerf_predict(X_test_sc[:1])
for _ in range(N_MEASURE):
    t = time.perf_counter()
    aerf_predict(X_test_sc[:1])
    times_aerf.append((time.perf_counter()-t)*1000)
lat_aerf = np.array(times_aerf)

latency_data = {
    'Logistic Regression': lat_lr,
    'Random Forest':       lat_rf,
    'CNN':                 lat_cnn,
    'LSTM':                lat_lstm,
    'Hybrid AE-RF':        lat_aerf,
}

print('\n=== Table 8: Inference Latency (ms/sample) ===')
for name, lat in latency_data.items():
    print(f'{name:22s}: mean={lat.mean():.3f} ms, std={lat.std():.3f} ms, p95={np.percentile(lat,95):.3f} ms')

# Figure 11: Latency Box Plots
fig, ax = plt.subplots(figsize=(10, 5))
data_plot = [v for v in latency_data.values()]
bp = ax.boxplot(data_plot, patch_artist=True, notch=False,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
colors_bp = ['#4E79A7','#F28E2B','#E15759','#76B7B2','#59A14F']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_xticklabels(list(latency_data.keys()), rotation=20, ha='right')
ax.set_ylabel('Inference Time (ms/sample)')
ax.set_title('Figure 11: Inference Latency Distribution Box Plots per Model', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure11_latency_boxplots.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 11 saved.')

## Cell 20: SHAP Explainability — Figures 12, 13, 14

In [ ]:
print('Running SHAP DeepExplainer on CNN...')

# Stratified sample of 1000 test points (smaller for speed/memory)
shap_n = min(1000, len(X_ts_cnn))
idx_shap = []
for cls in np.unique(y_test):
    cls_idx = np.where(y_test == cls)[0]
    n_take = max(1, int(shap_n * len(cls_idx) / len(y_test)))
    idx_shap.extend(np.random.choice(cls_idx, min(n_take, len(cls_idx)), replace=False))
idx_shap = np.array(idx_shap[:shap_n])

X_shap = X_ts_cnn[idx_shap]
y_shap = y_test[idx_shap]

# Background: 100 training samples
bg_idx = np.random.choice(len(X_tr_cnn), 100, replace=False)
background = X_tr_cnn[bg_idx]

explainer = shap.DeepExplainer(cnn_model, background)
shap_values = explainer.shap_values(X_shap)  # list[n_classes] or array

# Normalize shape: handle both (N, n_feat, 1) and (N, n_feat)
def squeeze_shap(sv):
    sv = np.array(sv)
    if sv.ndim == 3 and sv.shape[-1] == 1:
        return sv.squeeze(-1)       # (N, n_feat, 1) -> (N, n_feat)
    elif sv.ndim == 3:
        return sv.mean(axis=-1)     # (N, n_feat, ch) -> (N, n_feat)
    return sv                       # (N, n_feat) already

if isinstance(shap_values, list):
    shap_vals_squeezed = [squeeze_shap(sv) for sv in shap_values]
else:
    # packed as (N, n_feat, n_classes) or (N, n_feat)
    if shap_values.ndim == 3:
        shap_vals_squeezed = [shap_values[:, :, i] for i in range(shap_values.shape[-1])]
    else:
        shap_vals_squeezed = [shap_values]

print(f'SHAP: {len(shap_vals_squeezed)} class arrays, each shape {shap_vals_squeezed[0].shape}')

# Global mean absolute SHAP per feature
mean_abs_shap = np.mean([np.abs(sv) for sv in shap_vals_squeezed], axis=0)  # (N, n_feat)
global_importance = mean_abs_shap.mean(axis=0)                               # (n_feat,)
top_idx = np.argsort(global_importance)[::-1]
top_features = [selected_features[i] for i in top_idx[:15]]
top_values   = global_importance[top_idx[:15]]

print('Top 5 features by SHAP importance:')
for feat, val in zip(top_features[:5], top_values[:5]):
    print(f'  {feat}: {val:.4f}')

# Figure 12: Global SHAP Bar Plot
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_features[::-1], top_values[::-1],
        color=plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_features))))
ax.set_xlabel('Mean |SHAP Value| (Global Feature Importance)')
ax.set_title('Figure 12: SHAP Summary Plot — Global Feature Importance (CNN)', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure12_shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 12 saved.')

# Figure 13: SHAP Heatmap per Attack Class
n_cls = len(shap_vals_squeezed)
class_shap = np.array([np.abs(sv).mean(axis=0) for sv in shap_vals_squeezed])  # (n_cls, n_feat)
class_shap_top = class_shap[:, top_idx[:15]]
fig, ax = plt.subplots(figsize=(14, max(4, n_cls * 0.7)))
sns.heatmap(class_shap_top, xticklabels=top_features, yticklabels=CLASSES[:n_cls],
            cmap='YlOrRd', ax=ax, annot=False, linewidths=0.3)
ax.set_xlabel('Feature')
ax.set_ylabel('Attack Class')
ax.set_title('Figure 13: SHAP Feature Importance by Attack Class (Heatmap)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure13_shap_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 13 saved.')

# Figure 14: Per-class SHAP force-style bar plot
n_top = min(8, len(top_features))
fig, axes = plt.subplots(n_cls, 1, figsize=(12, n_cls * 2.2))
if n_cls == 1:
    axes = [axes]
for ci, (cls_name, ax) in enumerate(zip(CLASSES[:n_cls], axes)):
    sv = shap_vals_squeezed[ci]
    sample_mask = (y_shap == ci)
    if sample_mask.sum() == 0:
        ax.text(0.5, 0.5, f'{cls_name}: no samples', ha='center', va='center')
        ax.axis('off')
        continue
    sample_sv = sv[sample_mask][0]
    feat_vals = [(selected_features[i], float(sample_sv[i])) for i in range(len(selected_features))]
    feat_vals.sort(key=lambda x: abs(x[1]), reverse=True)
    feat_vals = feat_vals[:n_top]
    names = [f[0] for f in feat_vals]
    vals  = [f[1] for f in feat_vals]
    bar_colors = ['#E74C3C' if v > 0 else '#2980B9' for v in vals]
    ax.barh(names, vals, color=bar_colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(cls_name, fontsize=9, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)
fig.suptitle('Figure 14: SHAP Force Plot — Top Feature Contributions per Attack Class',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(FIG_DIR / 'figure14_shap_force_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 14 saved.')

## Cell 21: Deployment Architecture — Figure 15

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# IoT Devices layer
devices = ['Sensor\nLayer', 'IoT\nDevices', 'Industrial\nControllers']
for i, d in enumerate(devices):
    rect = plt.Rectangle((0.3 + i*2.2, 6.2), 1.7, 1.2, facecolor='#AED6F1',
                           edgecolor='#2980B9', linewidth=2)
    ax.add_patch(rect)
    ax.text(1.15 + i*2.2, 6.8, d, ha='center', va='center', fontsize=9, fontweight='bold')

# Gateway IDS
rect = plt.Rectangle((2, 4), 10, 1.6, facecolor='#D5E8D4', edgecolor='#27AE60', linewidth=2.5)
ax.add_patch(rect)
ax.text(7, 4.8, 'IoT Gateway — Lightweight CNN IDS Engine', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#145A32')
modules = ['Traffic\nCapture', 'Feature\nExtraction', 'Preprocessing\nPipeline',
           'CNN\nDetection', 'Alert\nManagement']
for i, m in enumerate(modules):
    x_m = 2.2 + i*2.0
    rect_m = plt.Rectangle((x_m, 4.05), 1.7, 0.7, facecolor='#A9DFBF',
                             edgecolor='#1E8449', linewidth=1)
    ax.add_patch(rect_m)
    ax.text(x_m + 0.85, 4.4, m, ha='center', va='center', fontsize=7.5)
    if i < len(modules)-1:
        ax.annotate('', xy=(x_m+1.7, 4.4), xytext=(x_m+1.95, 4.4),
                    arrowprops=dict(arrowstyle='->', lw=1.2, color='#145A32'))

# SIEM
rect = plt.Rectangle((2, 1.5), 4.5, 1.5, facecolor='#FDEBD0', edgecolor='#E67E22', linewidth=2)
ax.add_patch(rect)
ax.text(4.25, 2.25, 'SIEM Platform\n(Splunk / Elastic / QRadar)', ha='center', va='center',
        fontsize=9, fontweight='bold', color='#784212')

# SOC
rect = plt.Rectangle((7.5, 1.5), 4.5, 1.5, facecolor='#FDEDEC', edgecolor='#E74C3C', linewidth=2)
ax.add_patch(rect)
ax.text(9.75, 2.25, 'Security Operations Center\n(SOC Analyst Triage)', ha='center', va='center',
        fontsize=9, fontweight='bold', color='#922B21')

# Retraining loop
rect = plt.Rectangle((4.5, 0.2), 5, 0.9, facecolor='#EBF5FB', edgecolor='#5DADE2', linewidth=1.5)
ax.add_patch(rect)
ax.text(7, 0.65, 'Scheduled Retraining Pipeline (Model Drift Correction)', ha='center',
        va='center', fontsize=8.5, color='#1A5276')

# Arrows
for i in range(3):
    ax.annotate('', xy=(1.15 + i*2.2, 5.6), xytext=(1.15 + i*2.2, 6.2),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#555'))
ax.annotate('', xy=(4.25, 3.0), xytext=(7, 4.0),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#E67E22'))
ax.annotate('', xy=(9.75, 3.0), xytext=(7, 4.0),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#E74C3C'))
ax.annotate('', xy=(7, 1.1), xytext=(4.5, 1.5),
            arrowprops=dict(arrowstyle='<->', lw=1.2, color='#5DADE2'))

ax.set_title('Figure 15: Proposed IDS Deployment Architecture Diagram',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'figure15_deployment_architecture.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure 15 saved.')

## Cell 23: Summary — Fill in Paper Placeholder Values

In [ ]:
print('='*60)
print('RESULTS SUMMARY — use these to fill paper placeholders')
print('='*60)
print(f'CNN Accuracy:  {metrics_cnn["Accuracy"]*100:.2f}%')
print(f'CNN F1 Score:  {metrics_cnn["F1"]*100:.2f}%')
print(f'CNN ROC-AUC:   {metrics_cnn["ROC-AUC"]:.4f}')
print(f'CNN Latency (mean): {lat_cnn.mean():.3f} ms')
print()
print('Top 5 SHAP Features:')
for feat, val in zip(top_features[:5], top_values[:5]):
    print(f'  {feat}: {val:.4f}')
print()
print('All model F1 scores:')
for name, mvals in all_metrics.items():
    print(f'  {name:22s}: {mvals["F1"]*100:.2f}%')
print()
print(f'Figures saved to: {FIG_DIR}')
print(f'Models saved to:  {MODEL_DIR}')
print(f'Fixed document:   {OUT_PATH}')